# 03 — Per-Strike Minute-by-Minute Scraper

For each strike, click chart icon → export → download CSV to get intraday Call & Put LTP.

In [43]:
%run 01_setup.ipynb

Project root: C:\Users\sarda\Desktop\nse-options-scraper
Today: 2026-03-21


In [44]:
XPATH_CALL_CHART_ROW = '//img[@title="Calls chart for row {row}"]'
XPATH_PUT_CHART_ROW  = '//img[@title="Puts chart for row {row}"]'

XPATH_MODAL        = '//div[@id="modal_oc_graph"]'
XPATH_MODAL_CLOSE  = '//div[@id="modal_oc_graph"]//button[contains(@class, "close") or contains(@class, "btn-close")]'

In [45]:
def get_strike_rows(driver):
    """Get all row numbers that have chart icons in the option chain table."""
    rows = []
    row_num = 1
    while True:
        found = False
        for xpath in [XPATH_CALL_CHART_ROW.format(row=row_num), XPATH_PUT_CHART_ROW.format(row=row_num)]:
            try:
                driver.find_element(By.XPATH, xpath)
                found = True
            except NoSuchElementException:
                pass
        if not found:
            break
        rows.append(row_num)
        row_num += 1
    print(f"  Found {len(rows)} strike rows")
    return rows

In [46]:
def get_strike_from_csv(filepath):
    try:
        df = pd.read_csv(filepath)
        for col in df.columns:
            for part in str(col).strip().split():
                try:
                    val = float(part)
                    if val > 100:
                        return str(int(val)) if val == int(val) else str(val)
                except ValueError:
                    continue
    except Exception:
        pass
    return "unknown"

In [47]:
def close_chart_modal(driver):
    try:
        close_btn = driver.find_element(By.XPATH, XPATH_MODAL_CLOSE)
        safe_click(driver, close_btn)
        time.sleep(1)
    except NoSuchElementException:
        try:
            driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
            time.sleep(1)
        except Exception:
            pass

    try:
        WebDriverWait(driver, 5).until(
            EC.invisibility_of_element_located((By.XPATH, XPATH_MODAL))
        )
    except TimeoutException:
        driver.execute_script("""
            var modal = document.getElementById('modal_oc_graph');
            if (modal) { modal.style.display = 'none'; modal.classList.remove('show'); }
            document.querySelectorAll('.modal-backdrop').forEach(e => e.remove());
            document.body.classList.remove('modal-open');
            document.body.style.overflow = 'auto';
        """)
        time.sleep(0.5)

In [48]:
def get_strike_link(driver, row_num, side="call"):
    """Find the green grficon parent link for a strike row."""
    title_needle = f"{'Calls' if side == 'call' else 'Puts'} chart for row {row_num}"
    for link in driver.find_elements(By.CSS_SELECTOR, "a.js-oc-graph"):
        imgs = link.find_elements(By.TAG_NAME, "img")
        if imgs and title_needle in (imgs[0].get_attribute("title") or ""):
            return link
    return None


def get_strike_from_link(link):
    strike = link.get_attribute("data-oc-sp")
    if strike:
        try:
            val = float(strike.replace(",", ""))
            return str(int(val)) if val == int(val) else str(val)
        except ValueError:
            return strike
    return None


def get_strike_from_modal(driver):
    """Read strike price from the open chart modal header."""
    try:
        modal_text = driver.find_element(By.XPATH, XPATH_MODAL).text
        lines = [line.strip() for line in modal_text.split("\n") if line.strip()]
        for i, line in enumerate(lines):
            if "strike" in line.lower() and "price" in line.lower():
                if ":" in line and line.split(":", 1)[1].strip():
                    num = line.split(":", 1)[1].strip().replace(",", "")
                elif i + 1 < len(lines):
                    num = lines[i + 1].replace(",", "")
                else:
                    continue
                val = float(num)
                return str(int(val)) if val == int(val) else str(val)
    except Exception:
        pass
    return None


def download_strike_csv(driver, row_num, strike_map):
    """
    Click the green grficon link (parent <a>, not the <img>).
    The CSV contains BOTH call and put intraday ticks.
    Returns (downloaded_filepath, strike_price) or (None, None).
    """
    try:
        chart_link = get_strike_link(driver, row_num, side="call")
        if chart_link is None:
            chart_link = get_strike_link(driver, row_num, side="put")
        if chart_link is None:
            return None, None

    driver.execute_script('arguments[0].scrollIntoView({block: "center"});', chart_link)
    time.sleep(0.3)
    driver.execute_script("arguments[0].click();", chart_link)

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, XPATH_CHART_MENU))
        )
        time.sleep(2)
    except TimeoutException:
        print(f"    [WARN] Chart did not load for row {row_num}")
        close_chart_modal(driver)
        return None, None

    strike = strike_map.get(row_num) or get_strike_from_link(chart_link) or get_strike_from_modal(driver)
    if not strike:
        strike = f"row{row_num}"

    try:
        menu_btn = driver.find_element(By.XPATH, XPATH_CHART_MENU)
        driver.execute_script("arguments[0].dispatchEvent(new MouseEvent('click', {bubbles: true}));", menu_btn)
        time.sleep(1)
    except NoSuchElementException:
        print(f"    [WARN] No chart menu for row {row_num}")
        close_chart_modal(driver)
        return None, None

    try:
        WebDriverWait(driver, 5).until(
            lambda d: d.find_element(By.CSS_SELECTOR, "div.highcharts-contextmenu").value_of_css_property("display") != "none"
        )
        time.sleep(0.3)
    except TimeoutException:
        close_chart_modal(driver)
        return None, None

    before = set(os.listdir(TEMP_DOWNLOAD))
    driver.execute_script("""
        var items = document.querySelectorAll('.highcharts-contextmenu .highcharts-menu-item');
        for (var i = 0; i < items.length; i++) {
            if (items[i].textContent.trim() === 'Download CSV') {
                items[i].click();
                break;
            }
        }
    """)
    time.sleep(1)

    try:
        downloaded = wait_for_download(before, timeout=30)
    except TimeoutError:
        print(f"    [WARN] Download timeout for row {row_num}")
        close_chart_modal(driver)
        return None, None

    close_chart_modal(driver)
    return downloaded, strike

In [49]:
def scrape_strike_data(driver, symbol, symbol_type, expiry_text):
    select_symbol(driver, symbol, symbol_type)

    exp_dropdown = driver.find_element(By.XPATH, XPATH_EXPIRY_DROPDOWN)
    Select(exp_dropdown).select_by_visible_text(expiry_text)
    time.sleep(3)

    expiry_clean = expiry_text.replace(" ", "-").replace(",", "")
    dest_dir = os.path.join(DATA_RAW, symbol.lower(), "strike_data")
    os.makedirs(dest_dir, exist_ok=True)

    strike_map = get_strike_map(driver, symbol_type)
    rows = get_strike_rows(driver)
    print(f"  Scraping {len(rows)} rows for {symbol} | Expiry: {expiry_text}")

    for row_num in rows:
        clear_temp_downloads()
        downloaded, strike = download_strike_csv(driver, row_num, strike_map)

        if downloaded:
            dest_file = os.path.join(dest_dir, f"{TODAY}_{expiry_clean}_{strike}.csv")
            shutil.move(downloaded, dest_file)
            print(f"    Row {row_num:3d} | Strike {strike:>10s} | Saved")
        else:
            print(f"    Row {row_num:3d} | Skipped")

        time.sleep(1)

In [50]:
# Set symbol from watchlist; None = first symbol in config
SCRAPE_SYMBOL = "NIFTY"
SCRAPE_EXPIRY = None

driver = get_driver()
clear_temp_downloads()

try:
    init_nse_session(driver)

    watchlist = get_watchlist()
    item = next((w for w in watchlist if w["symbol"] == normalize_symbol(SCRAPE_SYMBOL)), watchlist[0])
    symbol = item["symbol"]
    symbol_type = item["type"]

    if SCRAPE_EXPIRY is None:
        select_symbol(driver, symbol, symbol_type)
        expiries = get_expiries_for_symbol(driver, symbol, symbol_type)
        SCRAPE_EXPIRY = expiries[0]
        print(f"Auto-selected nearest expiry: {SCRAPE_EXPIRY}")

    scrape_strike_data(driver, symbol, symbol_type, SCRAPE_EXPIRY)
    print(f"\nDone! Strike data saved to {DATA_RAW}")

except Exception as e:
    print(f"Error: {e}")
    raise

finally:
    driver.quit()
    print("Browser closed.")

Visiting NSE homepage for session cookies...
Navigating to Option Chain page...
Option chain page loaded.
Auto-selected nearest expiry: 24-Mar-2026
  [WARN] Could not read strikes from table, will extract from modal
  Found 135 strike rows
  Scraping 135 rows for NIFTY | Expiry: 24-Mar-2026
    Row   1 | Strike       row1 | Saved
    Row   2 | Strike       row2 | Saved
    Row   3 | Strike       row3 | Saved
    Row   4 | Strike       row4 | Saved
    Row   5 | Strike       row5 | Saved
    Row   6 | Strike       row6 | Saved
    Row   7 | Strike       row7 | Saved
    Row   8 | Strike       row8 | Saved
    Row   9 | Strike       row9 | Saved
    Row  10 | Strike      row10 | Saved
    Row  11 | Strike      row11 | Saved
    Row  12 | Strike      row12 | Saved
    Row  13 | Strike      row13 | Saved
    Row  14 | Strike      row14 | Saved
    Row  15 | Strike      row15 | Saved
    Row  16 | Strike      row16 | Saved
    Row  17 | Strike      row17 | Saved
    Row  18 | Strike      ro